# Phase 9 — Complete Model Comparison + Error Analysis + Final Model Selection

**Project**: Sarcasm & Irony Detection System using NLP  
**Degree**: MCA Final-Year Academic Capstone Project  
**Phase**: 9 — Holistic Evaluation & Production Model Selection  
**Dataset**: Leakage-Controlled Kaggle 4-Class Sarcasm & Irony Dataset (5,527 official test records)  
**Evaluated Approaches**:
1. Phase 4: Bag of Words (BoW) + LinearSVM
2. Phase 5: TF-IDF + Classical Classifiers (LinearSVM & Logistic Regression)
3. Phase 6: GloVe 100d Dense Word Embeddings + Bidirectional LSTM
4. Phase 7: FastText 300d Subword Synthesis + Bi-LSTM + Feed-Forward Attention
5. Phase 8: Pretrained BERT Contextual Transformer (`bert-base-uncased`)



## 1. Setup & Load Stored Experiment Results
We load the actual, stored test evaluation results across all project phases.



In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Add ml-service to path
sys.path.insert(0, os.path.abspath('..'))

sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.sans-serif": ["DejaVu Sans", "Arial"]})

# Paths
RESULTS_DIR = os.path.join('..', 'results', 'final_progression')
VIS_DIR = os.path.join(RESULTS_DIR, 'visualizations')

# Load stored CSV deliverables
df_candidates = pd.read_csv(os.path.join(RESULTS_DIR, 'final_candidates.csv'))
df_comparison = pd.read_csv(os.path.join(RESULTS_DIR, 'model_comparison.csv'))
df_per_class = pd.read_csv(os.path.join(RESULTS_DIR, 'per_class_comparison.csv'))
df_sarcasm_irony = pd.read_csv(os.path.join(RESULTS_DIR, 'sarcasm_irony_confusion.csv'))
df_complexity = pd.read_csv(os.path.join(RESULTS_DIR, 'model_complexity.csv'))
df_errors = pd.read_csv(os.path.join(RESULTS_DIR, 'error_analysis.csv'))

print("All Phase 9 result tables loaded successfully.")
print(f"Total candidates compared: {len(df_candidates)}")



## 2. Validate Result Consistency
Verify that:
- All models were evaluated on the exact same 5,527 official test samples
- The label mapping is strictly consistent across all phases: `Regular: 0, Sarcasm: 1, Irony: 2, Figurative: 3`
- No cross-split data leakage or fabricated values exist



In [ ]:
with open(os.path.join('..', 'data', 'processed', 'label_mapping.json')) as f:
    label_map = json.load(f)

test_df = pd.read_csv(os.path.join('..', 'data', 'processed', 'test_clean.csv'))

print(f"Verified Test Dataset Rows: {len(test_df)}")
print(f"Verified Label Mapping: {label_map}")
print("
Test Class Distribution:")
print(test_df['label'].value_counts())



## 3. Final Candidate Selection
The comparison focuses on the strongest candidate from each major NLP approach:
1. `CAND-01`: Best BoW + Classical ML (LinearSVM, Phase 4)
2. `CAND-02`: Best TF-IDF + Classical ML (LinearSVM, Phase 5)
3. `CAND-03`: Best TF-IDF + Calibrated Probabilities (LogisticRegression, Phase 5)
4. `CAND-04`: Best GloVe Dense Word Embeddings + Bi-LSTM (Phase 6)
5. `CAND-05`: Best FastText Subword Synthesis + Bi-LSTM (Phase 7)
6. `CAND-06`: Best FastText + Bi-LSTM + Feed-Forward Attention (Phase 7)
7. `CAND-07`: Best Contextual Transformer (BERT `bert-base-uncased`, Phase 8)



In [ ]:
display_cols = ['candidate_id', 'phase', 'model', 'representation', 'accuracy', 'macro_f1', 'training_time']
df_candidates[display_cols]



## 4. Overall Performance Comparison
The primary evaluation metric is **Macro F1** because our dataset involves four distinct classes with uneven distributions, and every class must receive equal importance.



In [ ]:
df_comparison.sort_values(by="Macro F1", ascending=False)



In [ ]:
# Visualize Macro F1 Comparison
plt.figure(figsize=(10, 5))
df_sorted = df_candidates.sort_values(by="macro_f1", ascending=True)
bars = plt.barh(df_sorted["model"], df_sorted["macro_f1"], color="#2b5c8f", height=0.55)
plt.axvline(0.70, color="#d9534f", linestyle="--", alpha=0.7, label="Macro F1 = 0.70 Reference")
plt.title("Official Test Set Macro F1 Comparison (Primary Metric)", fontsize=13, fontweight="bold")
plt.xlabel("Macro F1-Score")
plt.xlim(0.65, 0.75)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.0015, bar.get_y() + bar.get_height()/2, f"{w:.4f}", va="center", fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()



## 5. Per-Class Performance Comparison
Analyzing performance across individual classes reveals why a model with higher accuracy is not necessarily the superior classifier.



In [ ]:
df_pc_pivot = df_per_class.pivot(index="model", columns="class", values="f1")[["Regular", "Sarcasm", "Irony", "Figurative"]]
df_pc_pivot



In [ ]:
# Plot Class-wise F1 Comparison
df_pc_pivot.plot(kind="bar", figsize=(12, 6), colormap="viridis", width=0.75, edgecolor="#333333")
plt.title("Per-Class F1-Score Breakdown Across Model Architectures", fontsize=13, fontweight="bold")
plt.ylabel("F1-Score")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1.1)
plt.legend(title="Class", loc="upper right")
plt.tight_layout()
plt.show()



## 6. Confusion Matrix Comparison
Comparing the error distributions between the top classical model (LinearSVM) and the deep contextual transformer (BERT).



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
classes = ["Regular", "Sarcasm", "Irony", "Figurative"]

cm_bow = np.array([
    [1857, 1, 0, 0],
    [0, 1518, 0, 84],
    [1, 0, 1231, 69],
    [0, 469, 229, 68]
])
sns.heatmap(cm_bow, annot=True, fmt="d", cmap="Blues", xticklabels=classes, yticklabels=classes, ax=axes[0], cbar=False)
axes[0].set_title("CAND-01: LinearSVM (BoW)\nAccuracy: 84.57% | Macro F1: 0.7187", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Actual Label")

cm_bert = np.array([
    [1857, 1, 0, 0],
    [0, 1602, 0, 0],
    [0, 0, 1301, 0],
    [0, 525, 241, 0]
])
sns.heatmap(cm_bert, annot=True, fmt="d", cmap="Greens", xticklabels=classes, yticklabels=classes, ax=axes[1], cbar=False)
axes[1].set_title("CAND-07: BERT (bert-base-uncased)\nAccuracy: 86.12% | Macro F1: 0.6935", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("True Actual Label")

plt.suptitle("Side-by-Side Confusion Matrix Comparison", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()



## 7. Sarcasm vs. Irony Distinction Analysis
We examine whether models struggle to separate sarcasm from irony. Both models achieve near-flawless discrimination between the two classes.



In [ ]:
df_sarcasm_irony



## 8. Qualitative Error Analysis
Examining the distribution of linguistic error categories.



In [ ]:
print("Top Error Categories Across Models:")
print(df_errors['error_type'].value_counts())

print("
Sample Misclassified Tweets:")
df_errors[['text', 'actual_label', 'predicted_label', 'model', 'error_type']].head(8)



## 9. Confidence Analysis & Overconfident Errors
Analyzing cases where the model made an incorrect prediction with high confidence.



In [ ]:
overconfident = df_errors[(df_errors['actual_label'] != df_errors['predicted_label']) & (df_errors['confidence'] >= 0.90)]
print(f"Total High-Confidence Errors (Confidence >= 90%): {len(overconfident)}")
overconfident[['text', 'actual_label', 'predicted_label', 'model', 'confidence']].head(5)



## 10. Computational Complexity & Inference Benchmark
Comparing parameter counts, training durations, and latency measured over 1,000 standardized test tweets.



In [ ]:
df_complexity[['model', 'parameter_count', 'training_time_s', 'inference_latency_ms_per_text', 'inference_throughput_texts_per_s', 'memory_ram']]



In [ ]:
# Latency comparison plot
plt.figure(figsize=(10, 5))
df_lat = df_complexity.sort_values(by="inference_latency_ms_per_text", ascending=True)
bars = plt.barh(df_lat["model"], df_lat["inference_latency_ms_per_text"], color="#319795", height=0.55)
plt.xscale("log")
plt.title("Standardized Inference Latency Benchmark (Log Scale ms/text)", fontsize=13, fontweight="bold")
plt.xlabel("Latency in Milliseconds per Text")
for bar in bars:
    w = bar.get_width()
    plt.text(w * 1.15, bar.get_y() + bar.get_height()/2, f"{w:.3f} ms", va="center", fontweight="bold")
plt.tight_layout()
plt.show()



## 11. Model Ranking & Decision Framework
Evaluating candidates using the multi-criteria framework:
- Performance (Macro F1 & Accuracy): 40%
- Class Balance (avoiding minority collapse): 20%
- Computational Efficiency: 15%
- Inference Efficiency: 10%
- Interpretability: 10%
- Deployment Practicality: 5%



In [ ]:
ranking_data = {
    "Model": ["LinearSVM (BoW)", "LinearSVM (TF-IDF)", "GloVe + Bi-LSTM", "FastText + Attn", "BERT (bert-base)"],
    "Performance (40%)": [37, 36, 35, 35, 36],
    "Class Balance (20%)": [18, 12, 10, 9, 10],
    "Compute Efficiency (15%)": [15, 14, 11, 10, 7],
    "Inference Latency (10%)": [10, 10, 9, 8, 5],
    "Interpretability (10%)": [9, 9, 6, 7, 6],
    "Deployment Practicality (5%)": [5, 5, 4, 4, 3],
    "Total Score (100)": [94, 86, 75, 73, 67]
}
df_ranking = pd.DataFrame(ranking_data).sort_values(by="Total Score (100)", ascending=False)
df_ranking



## 12. Final Model Selection
**Selected Model**: `SarcasmIronyClassifier-LinearSVM-v1` (CAND-01: Bag of Words LinearSVM)

**Justification Summary**:
1. **Highest Macro F1 (0.7187)**: Superior overall score across all 4 classes.
2. **Class Balance Sensitivity**: Only architecture that did not collapse the minority Figurative class to 0.00% recall.
3. **Flawless Regular Text Separation**: 99.95% recall on literal text.
4. **Ultra-Low Latency**: 0.027 ms/text, enabling high-throughput microservices.



## 13. Reload Final Model from Disk
Verifying disk serialization into `models/final/` using the standalone predictor interface.



In [ ]:
from src.final_model.predict import SarcasmIronyPredictor

predictor = SarcasmIronyPredictor()
print(f"Loaded Final Model: {predictor.config.get('final_model_name')}")
print(f"Target Classes: {predictor.classes}")



## 14. Live Inference Testing
Executing single-text and batch predictions on diverse sample tweets.



In [ ]:
test_samples = [
    "The weather is sunny and clear today in San Francisco.",
    "Oh wonderful, another flat tire on Monday morning! Just what I needed. #sarcasm",
    "A fire station burning down while the firefighters are out on call.",
    "I have told you a million times, her smile was like sunshine in the dead of winter."
]

for text in test_samples:
    res = predictor.predict(text)
    print(f"Text: '{res['text']}'")
    print(f"--> Prediction: {res['predicted_class']} (Confidence: {res['confidence']:.2%})")
    print(f"--> Probabilities: {res['probabilities']}\n")



## 15. Final Conclusions & Transition to Phase 10
- **Academic Finding**: Contextual transformers (BERT) achieve high accuracy on majority classes and eliminate mutual Sarcasm-Irony confusion, but sparse linear classifiers (LinearSVM) optimize decision boundaries better on imbalanced minority classes, achieving higher Macro F1.
- **Production Finding**: LinearSVM provides sub-millisecond inference and minimal container footprints, making it the ideal engine for Phase 10 packaging and Phase 11 REST API deployment.
- **Phase 9 is COMPLETE**. Ready for Phase 10.

